In [12]:
import numpy as np
import nibabel as nib
from nilearn import datasets
import h5py
from pathlib import Path

Atlases

In [13]:
# 1. Fetch both Schaefer atlases (using 7 networks and 2mm resolution as the standard)
print("Downloading/Loading Schaefer atlases...")
atlas_1000 = datasets.fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=7, resolution_mm=2)
atlas_100 = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7, resolution_mm=2)

img_1000 = nib.load(atlas_1000.maps).get_fdata()
img_100 = nib.load(atlas_100.maps).get_fdata()

# 2. Build the spatial mapping dictionary
# We look at every voxel in the 1000 atlas and find what region it corresponds to in the 100 atlas
print("Building the spatial mapping...")
mapping_1000_to_100 = {}

for roi_1000 in range(1, 1001):
    # Find all voxels that belong to this specific ROI in the 1000-parcel atlas
    mask = (img_1000 == roi_1000)
    
    # Get the ROI values of those exact same voxels in the 100-parcel atlas
    overlapping_100_rois = img_100[mask]
    
    # Exclude the background (0)
    overlapping_100_rois = overlapping_100_rois[overlapping_100_rois != 0]
    
    if len(overlapping_100_rois) > 0:
        # Find the most frequent overlapping region (the mode)
        values, counts = np.unique(overlapping_100_rois, return_counts=True)
        best_match_100 = values[np.argmax(counts)]
        mapping_1000_to_100[roi_1000] = int(best_match_100)


Downloading/Loading Schaefer atlases...
[fetch_atlas_schaefer_2018] Dataset found in /home/fmoss/nilearn_data/schaefer_2018
[fetch_atlas_schaefer_2018] Dataset found in /home/fmoss/nilearn_data/schaefer_2018
Building the spatial mapping...


Load timeseries and compute FC

In [15]:
import h5py
import numpy as np
from pathlib import Path

# Set up the base directory and file mappings
data_dir = Path("../raw_data")

# Map the target class names and runs to their actual files
files_to_process = {
    "AP": {
        "REST1": data_dir / "hcpep_schaefer1000_REST1_AP.mat",
        "REST2": data_dir / "hcpep_schaefer1000_REST2_AP.mat"
    },
    "NAP": {
        "REST1": data_dir / "hcpep_schaefer1000_REST1_PA.mat",
        "REST2": data_dir / "hcpep_schaefer1000_REST2_PA.mat"
    }
}

# Nested dictionary to store raw results: raw_fc_results['class']['run']['metric']['subject_id']
raw_fc_results = {
    "AP": {"REST1": {"FC": {}, "FS": {}}, "REST2": {"FC": {}, "FS": {}}},
    "NAP": {"REST1": {"FC": {}, "FS": {}}, "REST2": {"FC": {}, "FS": {}}}
}

# Assumes `mapping_1000_to_100` from the previous Nilearn step is already in memory
# --- OPTIMIZATION: Precompute downsampling row indices once to save CPU time ---
roi_indices_100 = [
    [k - 1 for k, v in mapping_1000_to_100.items() if v == roi_100]
    for roi_100 in range(1, 101)
]

for class_label, runs in files_to_process.items():
    for run_name, file_path in runs.items():
        print(f"\n{'='*50}")
        print(f"Processing Class: {class_label} | Run: {run_name} | File: {file_path.name}")
        print(f"{'='*50}")
        
        if not file_path.exists():
            print(f"Warning: File not found {file_path}. Skipping run.")
            continue
            
        with h5py.File(file_path, 'r') as mat_data:
            subject_ids_refs = mat_data["data"]["subject_id"][:]
            ts_refs = mat_data["data"]["timeseries"][:]
            
            for i in range(len(subject_ids_refs)):
                # -----------------------------------------------------------------
                # 1. Extract Subject ID and Time Series Matrix
                # -----------------------------------------------------------------
                sub_ref = subject_ids_refs[i, 0] if subject_ids_refs.ndim > 1 else subject_ids_refs[i]
                sub_chars = mat_data[sub_ref][:]
                sub_name = ''.join(chr(c[0]) for c in sub_chars)
                
                ts_ref = ts_refs[i, 0] if ts_refs.ndim > 1 else ts_refs[i]
                dat_arr = mat_data[ts_ref][()].T  # Shape: (1000, timepoints)
                
                # -----------------------------------------------------------------
                # 2. Vectorized Voxelwise Robust Scaler (Median & IQR)
                # -----------------------------------------------------------------
                # No more slow python loops over 1000 voxels!
                medians = np.median(dat_arr, axis=1, keepdims=True)
                q25 = np.percentile(dat_arr, 25, axis=1, keepdims=True)
                q75 = np.percentile(dat_arr, 75, axis=1, keepdims=True)
                iqr = q75 - q25
                
                # Handle flat regions safely using vectorized conditions
                iqr = np.where(iqr == 0, 1e-6, iqr)
                
                # Apply normalization instantly across the whole matrix
                dat_arr = (dat_arr - medians) / iqr
                
                # -----------------------------------------------------------------
                # 3. Downsample from Schaefer 1000 to 100
                # -----------------------------------------------------------------
                timepoints = dat_arr.shape[1]
                ts_100 = np.zeros((100, timepoints))
                
                for idx, constituent_rows in enumerate(roi_indices_100):
                    if constituent_rows:
                        ts_100[idx, :] = np.mean(dat_arr[constituent_rows, :], axis=0)
                        
                # -----------------------------------------------------------------
                # 4. Compute Functional Connectivity and Shifted Functional Connectivity
                # -----------------------------------------------------------------
                # Contemporaneous FC
                fc_matrix = np.corrcoef(ts_100, rowvar=True)

                # Lagged/Shifted Functional Connectivity (FS) with tau = 1
                ts_current = ts_100[:, :-1]  # Source regions at time t
                ts_shifted = ts_100[:, 1:]   # Target regions at time t + 1

                # Stack to compute full cross-correlation quadrant safely
                combined_ts = np.vstack([ts_current, ts_shifted])
                full_corr = np.corrcoef(combined_ts, rowvar=True)

                # Extract upper right asymmetric quadrant
                num_regions = ts_100.shape[0]
                fs_matrix = full_corr[0:num_regions, num_regions:2*num_regions]
                
                # Store in our corrected dictionary structure
                raw_fc_results[class_label][run_name]["FC"][sub_name] = fc_matrix
                raw_fc_results[class_label][run_name]["FS"][sub_name] = fs_matrix
                
        # Fix print evaluation metrics to target the specific sub-dictionary length
        processed_count = len(raw_fc_results[class_label][run_name]["FC"])
        print(f"Successfully computed FC & FS matrices for {processed_count} subjects.")


Processing Class: AP | Run: REST1 | File: hcpep_schaefer1000_REST1_AP.mat
Successfully computed FC & FS matrices for 161 subjects.

Processing Class: AP | Run: REST2 | File: hcpep_schaefer1000_REST2_AP.mat
Successfully computed FC & FS matrices for 166 subjects.

Processing Class: NAP | Run: REST1 | File: hcpep_schaefer1000_REST1_PA.mat
Successfully computed FC & FS matrices for 165 subjects.

Processing Class: NAP | Run: REST2 | File: hcpep_schaefer1000_REST2_PA.mat
Successfully computed FC & FS matrices for 163 subjects.


In [16]:
# -----------------------------------------------------------------
# POST-PROCESSING: Filter unique subjects across runs (FC & FS)
# -----------------------------------------------------------------
print("\n" + "="*50)
print("POST-PROCESSING: Filtering Unique Subjects")
print("="*50)

# Final dictionary structure: final_fc_results['class']['metric']['subject_id']
final_fc_results = {
    "AP": {"FC": {}, "FS": {}},
    "NAP": {"FC": {}, "FS": {}}
}

# The order here defines priority (REST1 is checked/added first)
run_priority = ["REST1", "REST2"]

for class_label in final_fc_results.keys():
    duplicates_skipped = 0
    
    for run_name in run_priority:
        # Pull the specific FC and FS sub-dictionaries for this run
        run_fc_data = raw_fc_results[class_label][run_name].get("FC", {})
        run_fs_data = raw_fc_results[class_label][run_name].get("FS", {})
        
        # Iterate over subject IDs (using the FC dictionary keys)
        for sub_name in run_fc_data.keys():
            # Only add the subject if they aren't already in our final curated list
            if sub_name not in final_fc_results[class_label]["FC"]:
                final_fc_results[class_label]["FC"][sub_name] = run_fc_data[sub_name]
                final_fc_results[class_label]["FS"][sub_name] = run_fs_data[sub_name]
            else:
                duplicates_skipped += 1
                
    print(f"{class_label} Class:")
    print(f"  -> Total Unique Subjects Retained: {len(final_fc_results[class_label]['FC'])}")
    print(f"  -> Duplicates Skipped (from REST2): {duplicates_skipped}")

# Quick verification check on the final curated data
print("\n--- Final Summary ---")
for cls in final_fc_results:
    if final_fc_results[cls]["FC"]:
        sample_sub = list(final_fc_results[cls]["FC"].keys())[0]
        fc_shape = final_fc_results[cls]["FC"][sample_sub].shape
        fs_shape = final_fc_results[cls]["FS"][sample_sub].shape
        print(f"Final {cls} Class | Sample FC Matrix Shape: {fc_shape}")
        print(f"Final {cls} Class | Sample FS Matrix Shape: {fs_shape}")


POST-PROCESSING: Filtering Unique Subjects
AP Class:
  -> Total Unique Subjects Retained: 166
  -> Duplicates Skipped (from REST2): 161
NAP Class:
  -> Total Unique Subjects Retained: 166
  -> Duplicates Skipped (from REST2): 162

--- Final Summary ---
Final AP Class | Sample FC Matrix Shape: (100, 100)
Final AP Class | Sample FS Matrix Shape: (100, 100)
Final NAP Class | Sample FC Matrix Shape: (100, 100)
Final NAP Class | Sample FS Matrix Shape: (100, 100)


Clinical scores

In [31]:
from scipy.io import loadmat

clinical = loadmat("../raw_data/panss_id_scores.mat") 
ids = clinical["panss_id"]
scores = clinical["panss_score"]

In [34]:
from scipy.io import loadmat
import numpy as np

# =================================================================
# 1. LOAD CLINICAL DATA & BUILD LOOKUP DICTIONARY
# =================================================================
clinical = loadmat("../raw_data/panss_id_scores.mat") 
ids = clinical["panss_id"]
scores = clinical["panss_score"]

# Create a clean flat dictionary: {"1015_01_MR": 53, ...}
panss_lookup = {}
for i in range(len(ids)):
    # Safely extract scalar integers from the 2D array structure
    p_id = ids[i][0]
    p_score = scores[i][0]
    
    # Construct the full string key to match your H5/Mat subject format
    full_sub_name = f"{int(p_id)}_01_MR"
    panss_lookup[full_sub_name] = int(p_score)

print(f"Loaded {len(panss_lookup)} total clinical scores from MAT file.")

# =================================================================
# 2. ALIGN CLINICAL SCORES WITH CURATED IMAGING SUBJECTS
# =================================================================
print("\n" + "="*50)
print("ALIGNING PANSS SCORES TO RETAINED IMAGING SUBJECTS")
print("="*50)

for class_label in final_fc_results.keys():
    # Initialize a clean home for the clinical scores inside your structured dict
    final_fc_results[class_label]["PANSS"] = {}
    missing_scores_count = 0
    
    # Iterate through the actual unique subjects that survived deduplication
    for sub_name in final_fc_results[class_label]["FC"].keys():
        if sub_name in panss_lookup:
            final_fc_results[class_label]["PANSS"][sub_name] = panss_lookup[sub_name]
        else:
            # Fallback if a subject has imaging files but no recorded PANSS entry
            final_fc_results[class_label]["PANSS"][sub_name] = None
            missing_scores_count += 1
            
    total_matched = len(final_fc_results[class_label]["PANSS"]) - missing_scores_count
    print(f"{class_label} Class Alignment Metrics:")
    print(f"  -> Successfully Matched Scores: {total_matched}")
    print(f"  -> Missing Scores (Flagged None): {missing_scores_count}")

# =================================================================
# 3. VERIFICATION SANITY CHECK
# =================================================================
print("\n--- Unified Data Sample Check ---")
for cls in final_fc_results:
    if final_fc_results[cls]["FC"]:
        sample_sub = list(final_fc_results[cls]["FC"].keys())[0]
        sample_score = final_fc_results[cls]["PANSS"][sample_sub]
        print(f"Sub [{sample_sub}] in Class [{cls}] | PANSS Score: {sample_score} | FC Matrix Shape: {final_fc_results[cls]['FC'][sample_sub].shape}")

Loaded 181 total clinical scores from MAT file.

ALIGNING PANSS SCORES TO RETAINED IMAGING SUBJECTS
AP Class Alignment Metrics:
  -> Successfully Matched Scores: 107
  -> Missing Scores (Flagged None): 59
NAP Class Alignment Metrics:
  -> Successfully Matched Scores: 108
  -> Missing Scores (Flagged None): 58

--- Unified Data Sample Check ---
Sub [1001_01_MR] in Class [AP] | PANSS Score: None | FC Matrix Shape: (100, 100)
Sub [1001_01_MR] in Class [NAP] | PANSS Score: None | FC Matrix Shape: (100, 100)


In [39]:
final_fc_results["AP"]["PANSS"]["1015_01_MR"]

53

In [40]:
import pickle
from pathlib import Path

# Define the destination path
output_file = Path("../raw_data/final_curated_imaging_clinical_data.pkl")

print(f"Serializing and saving dataset to: {output_file}")

# Save the entire nested dictionary structure seamlessly
with open(output_file, 'wb') as f:
    pickle.dump(final_fc_results, f, protocol=pickle.HIGHEST_PROTOCOL)

print("🎉 Success! All FC matrices, FS matrices, and PANSS scores are securely saved in a single file.")

Serializing and saving dataset to: ../raw_data/final_curated_imaging_clinical_data.pkl
🎉 Success! All FC matrices, FS matrices, and PANSS scores are securely saved in a single file.
